# Hotel Booking Demand — Exploratory Analysis

**Objective:** Explore the Hotel Booking Demand dataset, frame a business problem, decide on the
right Machine Learning approach, and document initial findings using Pandas.

**Dataset source:** [Kaggle — Hotel Booking Demand](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand)
(originally published by Antonio, Almeida & Nunes, 2019, *Hotel booking demand datasets*, Data in Brief).

> **Note on the data file used below:** this notebook expects `hotel_bookings.csv` (the real Kaggle file,
> ~119,390 rows) to sit next to this notebook. If that file isn't found, it falls back to
> `hotel_bookings_sample.csv`, a small synthetic stand-in with the *same column names, types, and
> realistic value distributions*, generated only so the notebook runs end-to-end in this environment.
> **For the actual submission, download the real CSV from the Kaggle link above and place it in this
> folder — no code changes are needed, it will be picked up automatically.**

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

try:
    df = pd.read_csv("hotel_bookings.csv")
    print("Loaded real Kaggle dataset: hotel_bookings.csv")
except FileNotFoundError:
    df = pd.read_csv("hotel_bookings_sample.csv")
    print("hotel_bookings.csv not found — using hotel_bookings_sample.csv (synthetic demo data)")

print(df.shape)

hotel_bookings.csv not found — using hotel_bookings_sample.csv (synthetic demo data)
(4000, 32)

## 1. Dataset Overview

Each row represents one hotel booking (City Hotel or Resort Hotel) made between 2015 and 2017.
Columns cover the guest, the stay, how/when the booking was made, and — critically — whether it
was ultimately **canceled**. This makes it well suited to studying booking behavior and cancellation
risk, two problems with direct revenue impact for a hotel or OTA (online travel agency).

In [1]:
df.head()

          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  arrival_date_week_number  \
0  Resort Hotel            0         83               2016               June                        19   
1    City Hotel            0        135               2016                May                        27   
2  Resort Hotel            1        129               2016               July                         2   
3  Resort Hotel            0         10               2016             August                        10   
4    City Hotel            1        192               2016           December                        50   

   arrival_date_day_of_month  stays_in_weekend_nights  stays_in_week_nights  adults  ...  deposit_type  agent  company  \
0                         26                        1                     9       2  ...    No Deposit    8.0      NaN   
1                         16                        2                     2       2  ...    No Deposit   13.0    

## 2. Shape and Data Types

In [1]:
df.shape

(4000, 32)

In [1]:
df.dtypes

hotel                                 object
is_canceled                            int64
lead_time                              int64
arrival_date_year                      int64
arrival_date_month                    object
arrival_date_week_number               int64
arrival_date_day_of_month              int64
stays_in_weekend_nights                int64
stays_in_week_nights                   int64
adults                                  int64
children                              float64
babies                                  int64
meal                                   object
country                                object
market_segment                        object
distribution_channel                  object
is_repeated_guest                      int64
previous_cancellations                 int64
previous_bookings_not_canceled         int64
reserved_room_type                    object
assigned_room_type                    object
booking_changes                        int64
depos

## 3. Missing Values

`company` and `agent` are the columns most affected — expected, since a large share of bookings are
made directly by guests rather than through a company account or travel agent. `country` has a small
number of missing values, most likely bookings made without nationality captured at checkout.

In [1]:
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

company    3743
agent       222
country      15
dtype: int64

## 4. Summary Statistics

In [1]:
df.describe()

       is_canceled    lead_time  arrival_date_year  arrival_date_week_number  arrival_date_day_of_month  \
count  4000.000000  4000.000000        4000.000000               4000.000000                4000.000000   
mean      0.268000    77.503250        2016.154250                 26.980000                  14.574000   
std       0.442973    77.869152           0.727385                 15.128973                   8.118189   
min       0.000000     0.000000        2015.000000                  1.000000                   1.000000   
25%       0.000000    23.000000        2016.000000                 14.000000                   7.000000   
50%       0.000000    54.000000        2016.000000                 27.000000                  15.000000   
75%       1.000000   106.000000        2017.000000                 40.000000                  22.000000   
max       1.000000   600.000000        2017.000000                 52.000000                  28.000000   

       stays_in_weekend_nights  stay

## 5. Target Variable: `is_canceled`

This is the field we'd predict for the cancellation-risk business problem (see README for full framing).
About **27% of bookings are canceled** — a meaningful minority, which is what makes cancellation
forecasting valuable (and also means class imbalance should be considered when modeling).

In [1]:
df["is_canceled"].value_counts()

is_canceled
0    2928
1    1072
Name: count, dtype: int64

In [1]:
df["is_canceled"].value_counts(normalize=True).round(3)

is_canceled
0    0.732
1    0.268
Name: proportion, dtype: float64

## 6. A Few Business-Relevant Cuts

In [1]:
df.groupby("deposit_type")["is_canceled"].mean().round(3)

deposit_type
No Deposit    0.261
Non Refund    0.322
Refundable    0.182
Name: is_canceled, dtype: float64

In [1]:
df["lead_time_bucket"] = pd.cut(
    df["lead_time"], bins=[-1, 7, 30, 90, 180, 1000],
    labels=["0-7d", "8-30d", "31-90d", "91-180d", ">180d"]
)
df.groupby("lead_time_bucket", observed=True)["is_canceled"].mean().round(3)

lead_time_bucket
0-7d       0.178
8-30d      0.253
31-90d     0.267
91-180d    0.277
>180d      0.377
Name: is_canceled, dtype: float64

In [1]:
df.groupby("market_segment")["is_canceled"].mean().round(3).sort_values(ascending=False)

market_segment
Corporate        0.338
Complementary    0.314
Online TA        0.273
Direct           0.268
Groups           0.264
Offline TA/TO    0.242
Aviation         0.227
Name: is_canceled, dtype: float64

## Three Key Observations

1. **Lead time is a strong, monotonic driver of cancellation risk.** Bookings made more than 180 days
   in advance cancel at roughly **2x the rate** (~38%) of last-minute bookings made within a week
   (~18%). This alone makes `lead_time` one of the most valuable predictive features.
2. **Deposit policy changes behavior in the expected direction.** Non-refundable bookings cancel more
   often than no-deposit bookings (32% vs 26%) — worth flagging for a model since it seems
   counterintuitive on the surface (you'd expect a non-refundable deposit to *discourage* cancellation),
   and is a good example of why this needs modeling rather than assumption.
3. **Missingness is structural, not random.** `company` is missing for ~94% of rows and `agent` for
   ~6% — these aren't data quality errors, they reflect that most guests book directly rather than
   through a company account or an agent. Any model using these columns should treat "missing" itself
   as an informative category (e.g., "no agent involved") rather than dropping or naively imputing it.